In [ ]:

import os, json
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

DATA = 'processed_features'
MODELS = 'wake_word_data/models'
os.makedirs(MODELS, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'



In [ ]:
def load(dataset,data):
  d = np.load(f'{data}/{dataset}.npz')
  return d['X'], d['y'], d['words']


Xtrain, ytrain, wtrain = load('train',DATA)
Xtest, ytest, wtest = load('test',DATA)
Xval, yval, wval = load('val',DATA)



In [ ]:
def loadToModel(X,y,shuffle):
  #pair each spectrogram with its label
  dataset = TensorDataset(torch.tensor(X).unsqueeze(1),torch.tensor(y)) #gives an explicit channel axis (1, 40, 101)
  #feeds model in batches of 64
  return DataLoader(dataset,batch_size=64,shuffle=shuffle)

train_dataloader = loadToModel(Xtrain,ytrain,True) # training is random shuffled
test_dataloader = loadToModel(Xtest,ytest,False)
val_dataloader = loadToModel(Xval,yval,False)



In [ ]:
class NeuralNet(nn.Module):
  def __init__(self, channel_widths=(32,64,64), num_classes=3):
    super().__init__()
    layers, channel_in = [], 1
    for i, c in enumerate(channel_widths):
      layers += [nn.Conv2d(channel_in,c,3, padding=1),nn.BatchNorm2d(c),nn.ReLU()]
      if i < len(channel_widths) -1:
        layers.append(nn.MaxPool2d(2))
      channel_in = c
    layers.append(nn.AdaptiveAvgPool2d(1))
    self.features = nn.Sequential(*layers)
    self.classifier = nn.Sequential(nn.Dropout(0.3),nn.Linear(channel_widths[-1],num_classes))
  
  def forward(self,x):
    return self.classifier(self.features(x).flatten(1))






In [ ]:
from sklearn.metrics import confusion_matrix


#need function to evaluate models performance
#talk about messing with threshold settings
def evaluate(model, dataloader, loss_function,wake,threshold=.4):
  model.eval()
  predictions, ground_truth_labels = [], []
  #dont need gradients for evaluation
  with torch.no_grad():
    for x,y in dataloader:
      x,y = x.to(device), y.to(device)
      predictions.append(model(x))
      ground_truth_labels.append(y)

  predictions = torch.cat(predictions)
  ground_truth_labels = torch.cat(ground_truth_labels)
  loss = loss_function(predictions,ground_truth_labels).item()

  probabilities = torch.softmax(predictions,dim=1)
  wake_prob = probabilities[:,wake]
  actual_wake_probs = wake_prob[ground_truth_labels == wake]

  print(
    f"Wake probabilities: "
    f"min={actual_wake_probs.min().item():.3f}, "
    f"max={actual_wake_probs.max().item():.3f}, "
    f"mean={actual_wake_probs.mean().item():.3f}"
  )

  pred = wake_prob >= threshold
  #talk about why i added softmax and threshold

  is_wake_word = ground_truth_labels == wake
  accuracy = (pred == is_wake_word).float().mean().item()
  fnr = (~pred[is_wake_word]).float().mean().item()
  fpr = (pred[~is_wake_word]).float().mean().item()

  cm = confusion_matrix(is_wake_word.cpu().numpy(),pred.cpu().numpy(),labels=[False,True])
  
  return loss, accuracy, fnr, fpr, cm

      
def train(model, name, epochs=30, learning_rate = .0003,wake_word=2):
  
  #talk about how learning rate was originally .001 in report
  model.to(device)
  optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)
  #talk about adding the scheduler in report
  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs)
  loss_function = nn.CrossEntropyLoss()
  best_loss = float('inf')

  #for saving history
  history = {'train_loss':[],'val_loss':[],'val_acc':[],'val_fnr':[],'val_fpr':[]}
  for epoch in range(epochs):
    model.train()
    running, n = 0, 0
    for x,y in train_dataloader:
      x,y = x.to(device), y.to(device)
      optimizer.zero_grad() #clears prev batch gradients
      loss = loss_function(model(x),y)
      loss.backward() #computes gradients
      optimizer.step() #updates weights
      
      running += loss.item()*len(y)
      n += len(y)
    train_loss = running/n
    history['train_loss'].append(train_loss)  
    val_loss, val_acc, val_fnr, val_fpr, cm = evaluate(model, val_dataloader, loss_function, wake_word)
    print(f'ep {epoch+1:2d} train_loss {train_loss:.3f} val loss {val_loss:.3f}  acc {val_acc:.3f}  FPR {val_fpr:.3f}  FNR {val_fnr:.3f}')
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_fnr'].append(val_fnr)
    history['val_fpr'].append(val_fpr)
    if epoch % 5 == 0:
      print(cm)
    scheduler.step()
    if val_loss < best_loss:
      best_loss = val_loss
      torch.jit.script(model).save(f'{MODELS}/{name}.pt')
  return best_loss,history


In [ ]:



#model = NeuralNet(channel_widths=(32,64,64))
#best_val_loss,run_history =train(model,'baseline')

PLOTS = 'wake_word_data/plots'
os.makedirs(PLOTS,exist_ok=True)
RESULTS = 'wake_word_data/results'
os.makedirs(RESULTS,exist_ok=True)

def run(name, widths, epochs=30):
  torch.manual_seed(0)
  model = NeuralNet(channel_widths=widths)
  num_params = sum(p.numel() for p in model.parameters())
  best_loss,run_history =train(model,name,epochs=epochs)
  run_history.update(best_val_lost=best_loss,best_epoch=int(np.argmin(run_history['val_loss']))+1,num_params=num_params)
  with open(f'{RESULTS}/{name}_history.json','w') as f:
    json.dump(run_history,f)
  return run_history


two_layer = run('two_layer',(32,64))






In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 16})

def plot_graphs(history,name):
  epoch = range(1,len(history['train_loss'])+1)
  plt.figure(figsize=(12,8))
  plt.plot(epoch,history['train_loss'],'b',label='Training loss')
  plt.plot(epoch,history['val_loss'],'r',label='Validation loss')
  plt.title(f'Training and validation loss {name}')
  plt.xlabel('Epochs')
  plt.ylabel('Loss')
  plt.legend()
  plt.savefig(f'{PLOTS}/{name}_loss.png',dpi=200,bbox_inches='tight')
  plt.show()


#plot_graphs(base,'baseline')

In [ ]:
plot_graphs(two_layer,'two_layer')

In [ ]:
three_layer_less_channels = run('three_layer_less_channels',(16,32,32))
plot_graphs(three_layer_less_channels,'three_layer_less_channels')

In [ ]:
two_layer_less_channels = run('two_layer_less_channels',(16,32))
plot_graphs(two_layer_less_channels,'two_layer_less_channels')

In [ ]:
base = run('base',(32,64,64))
plot_graphs(base,'base')

In [ ]:
!zip -r output_folder.zip /kaggle/working/wake_word_data